# A language model as a cloud service (Azure AI Foundry)

In the LLM basics notebook we ran tiny models on a laptop. The big ones don't fit on any laptop – you use them **as a service**: the model runs in a data center, you send text over HTTPS and pay per token.

Here we call **gpt-4.1-mini** deployed in our Azure AI Foundry resource (Sweden Central, "Data Zone EU" deployment – the data stays in the EU).

**Authentication without keys:** instead of an API key pasted into the code (which always ends up on GitHub…), we log in with our Azure account (`az login`) and Azure gives us a short-lived token. Access is granted by a role (*Cognitive Services OpenAI User*) on the resource.

## Setup

In [ ]:
%pip install -q openai azure-identity

**Two settings** (Azure portal → your Foundry resource → *Overview*): the **endpoint** and your **tenant (directory) ID**. Set them as environment variables (`AZURE_OPENAI_ENDPOINT`, `AZURE_TENANT_ID`) – or just paste them when the cell asks. They're not secret, but we keep them out of a public repository anyway.

- **In Google Colab** there's no `az login`, so we log in with a **device code**: the cell prints a link and a code – open the link, enter the code, log in (with MFA).
- **On your own computer** with the Azure CLI installed, run `az login` once in the terminal and the notebook reuses that login.

In [ ]:
import os
import sys
from azure.identity import AzureCliCredential, DeviceCodeCredential, get_bearer_token_provider
from openai import AzureOpenAI

ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT") or input("Foundry endpoint (https://....cognitiveservices.azure.com/): ").strip()
DEPLOYMENT = "gpt-4.1-mini"

if "google.colab" in sys.modules:
    tenant = os.environ.get("AZURE_TENANT_ID") or input("Tenant (directory) ID: ").strip()
    credential = DeviceCodeCredential(tenant_id=tenant)   # prints a link + code to log in
else:
    credential = AzureCliCredential()                     # reuses `az login`

token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")
client = AzureOpenAI(azure_endpoint=ENDPOINT, azure_ad_token_provider=token_provider, api_version="2024-10-21")

## 1. One question, one answer

Under the hood it's the same as in the LLM basics notebook: the messages are put into a chat template, and the model predicts the answer token by token. We only see the result.

In [ ]:
response = client.chat.completions.create(
    model=DEPLOYMENT,
    messages=[
        {"role": "system", "content": "You are a patient teacher of deep learning. Answer in 2-3 sentences."},
        {"role": "user", "content": "Why do neural networks train faster on GPUs than on CPUs?"},
    ],
)
print(response.choices[0].message.content)

## 2. Tokens = money

Every call reports how many tokens went in (the prompt) and out (the answer). That's what you pay for. Prices change, so check the [Azure OpenAI pricing page](https://azure.microsoft.com/pricing/details/cognitive-services/openai-service/) – the numbers below are just placeholders to show the calculation.

In [ ]:
PRICE_PER_1M_INPUT = 0.40     # USD, placeholder - check the pricing page
PRICE_PER_1M_OUTPUT = 1.60    # USD, placeholder

u = response.usage
cost = u.prompt_tokens / 1e6 * PRICE_PER_1M_INPUT + u.completion_tokens / 1e6 * PRICE_PER_1M_OUTPUT
print(f"prompt: {u.prompt_tokens} tokens, answer: {u.completion_tokens} tokens  →  ~${cost:.6f}")
print(f"The same call for a class of 30 students, 50 questions each: ~${cost * 30 * 50:.2f}")

## 3. Temperature – again

Same knob as in the LLM basics notebook: low temperature → predictable, high → creative (and eventually chaotic).

In [ ]:
prompt = [{"role": "user", "content": "Invent a name for a startup that sells GPUs to students. Just the name."}]
for t in [0.0, 1.0, 1.8]:
    names = [client.chat.completions.create(model=DEPLOYMENT, messages=prompt, temperature=t, max_tokens=15)
             .choices[0].message.content.strip() for _ in range(3)]
    print(f"temperature {t}: {names}")

## 4. Streaming

Chat apps show the answer word by word. That's not an animation – the tokens really arrive one after another, as the model generates them.

In [ ]:
stream = client.chat.completions.create(
    model=DEPLOYMENT, stream=True,
    messages=[{"role": "user", "content": "Explain cloud computing to a 10-year-old in 4 sentences."}],
)
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

## 5. Tool calling – the agent harness, with a big model

Remember the agent harness notebook? The 0.5B model was a weak agent. Here's the same idea with a much bigger model: we describe a tool, the model decides to call it, **our code runs it**, and we send the result back.

In [ ]:
import json

def gpu_price(hours: float, price_per_hour: float) -> float:
    """Our 'tool': cost of renting a machine."""
    return round(hours * price_per_hour, 2)

tools = [{
    "type": "function",
    "function": {
        "name": "gpu_price",
        "description": "Compute the cost of renting a cloud machine for some hours.",
        "parameters": {"type": "object",
                       "properties": {"hours": {"type": "number"}, "price_per_hour": {"type": "number"}},
                       "required": ["hours", "price_per_hour"]},
    },
}]

messages = [{"role": "user", "content": "A GPU VM costs $0.90 per hour. How much for training 3 models, 7.5 hours each?"}]
reply = client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools).choices[0].message

while reply.tool_calls:                                  # the harness loop
    messages.append(reply)
    for call in reply.tool_calls:
        args = json.loads(call.function.arguments)
        result = gpu_price(**args)
        print(f"[tool] {call.function.name}({args}) -> {result}")
        messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})
    reply = client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools).choices[0].message

print("[model]", reply.content)

## Your turn

1. Ask the same question with three different system prompts (a teacher, a pirate, a lawyer). How much does the answer change?
2. Find a question where `gpt-4.1-mini` is confidently wrong. (Hint: counting letters, very recent events, exact quotes.)
3. Compare the cost of answering 1 000 questions with this model vs. renting a GPU VM for a day to run an open model yourself. When does which make sense?